# Polyp Detection — Video Tracking

Demonstrates real-time polyp detection and tracking using the final
model (aug_neg) on a temporal sequence of colonoscopy frames.

---

## Approach

CVC-ClinicDB frames (1.png, 2.png, ...) were extracted from a real
colonoscopy video and retain their original temporal order. We assemble
them into an mp4 file and run the final model with ByteTrack, which
assigns consistent IDs to each polyp across frames.

This avoids any copyright or privacy issues associated with downloading
external colonoscopy videos, while still demonstrating tracking behavior
on real clinical data that we already hold a license for.

---

## What this notebook does

1. Sorts CVC-ClinicDB frames numerically to preserve temporal order
2. Assembles frames 1–150 into a 15-second mp4 at 10 FPS
3. Runs YOLOv8m-seg + ByteTrack on the assembled video
4. Reports per-frame detection stats and unique track IDs

## Results

| Metric | Value |
|--------|-------|
| Frames processed | 150 |
| Frames with detection | 126 (84%) |
| Unique track IDs | 14 |
| Inference speed | ~45 FPS |

> Note: unique track IDs represent an upper bound on distinct polyps,
> not an exact count — ByteTrack assigns a new ID when a polyp
> disappears for several frames and reappears.

## Output

- results/videos/cvc_input.mp4
- results/videos/cvc_tracked.mp4
- results/metrics/tracking_stats.json

In [ ]:
# Import libraries

import json
import os
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from ultralytics import YOLO

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    "model_weights": os.path.join(BASE_DIR, "models", "aug_neg", "weights", "best.pt"),
    "cvc_images":    os.path.join(BASE_DIR, "data", "yolo_format", "cvc_test", "images"),
    "videos":        os.path.join(BASE_DIR, "results", "videos"),
    "metrics":       os.path.join(BASE_DIR, "results", "metrics"),
}

os.makedirs(PATHS["videos"],  exist_ok=True)
os.makedirs(PATHS["metrics"], exist_ok=True)

# Video settings
VIDEO_FPS    = 10    # CVC frames are stills, not a live stream -
                     # 10 FPS gives a natural colonoscopy viewing speed
VIDEO_FRAMES = 150   # Use first 150 frames (covers most polyp types)
CONF_THRESH  = 0.30  # Same operating threshold selected in notebook 04

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:14s} -> {path}")

print(f"\nVideo settings:")
print(f"  FPS:    {VIDEO_FPS}")
print(f"  Frames: {VIDEO_FRAMES}")
print(f"  Conf:   {CONF_THRESH}")

In [ ]:
# Sort Frames Numerically
# CVC frames are named 1.png, 2.png, ..., 612.png
# We sort them numerically (not lexicographically) to preserve
# the original temporal order from the colonoscopy video

all_frames = sorted(
    [f for f in os.listdir(PATHS["cvc_images"])
     if f.lower().endswith((".jpg", ".jpeg", ".png"))],
    key=lambda x: int(os.path.splitext(x)[0])
)

selected_frames = all_frames[:VIDEO_FRAMES]

print(f"Total CVC frames available: {len(all_frames)}")
print(f"Frames selected for video:  {len(selected_frames)}")
print(f"First frame: {selected_frames[0]}")
print(f"Last frame:  {selected_frames[-1]}")

# Verify they are actually in numeric order
frame_numbers = [int(os.path.splitext(f)[0]) for f in selected_frames]
print(f"Frame number range: {frame_numbers[0]} to {frame_numbers[-1]}")

In [ ]:
# Build Input Video
# Assemble the selected frames into an mp4 file.
# We read the first frame to get the correct resolution.

input_video_path = os.path.join(PATHS["videos"], "cvc_input.mp4")

first_img = cv2.imread(os.path.join(PATHS["cvc_images"], selected_frames[0]))
height, width = first_img.shape[:2]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(input_video_path, fourcc, VIDEO_FPS, (width, height))

for fname in selected_frames:
    img = cv2.imread(os.path.join(PATHS["cvc_images"], fname))
    if img is not None:
        writer.write(img)

writer.release()

size_mb = os.path.getsize(input_video_path) / 1e6
print(f"Input video saved -> {input_video_path}")
print(f"Resolution: {width} x {height}")
print(f"Frames: {len(selected_frames)}")
print(f"Duration: {len(selected_frames) / VIDEO_FPS:.1f} seconds")
print(f"File size: {size_mb:.1f} MB")

In [ ]:
# Load Model
# Load the final model (aug_neg) - best configuration from experiments

print(f"Loading model: {PATHS['model_weights']}")
model = YOLO(PATHS["model_weights"])
print("Model loaded successfully.")

In [ ]:
# Run Tracking
# Run YOLOv8m-seg with ByteTrack on the input video.
# ByteTrack assigns a consistent ID to each polyp and maintains
# that ID across frames even when the polyp briefly disappears.
#
# persist=True tells the tracker to keep track state between frames,
# which is what enables consistent IDs across the video.

output_video_path = os.path.join(PATHS["videos"], "cvc_tracked.mp4")

print("Running detection + tracking...")
start_time = time.time()

track_results = model.track(
    source=input_video_path,
    conf=CONF_THRESH,
    tracker="bytetrack.yaml",
    persist=True,
    imgsz=640,
    save=True,
    project=PATHS["videos"],
    name="tracked_output",
    exist_ok=True,
)

elapsed = time.time() - start_time
print(f"Tracking finished in {elapsed:.1f} seconds")

In [ ]:
# Analyze Tracking Results
# Compute per-frame statistics from the tracking results

frame_detections = []
all_track_ids     = set()

for result in track_results:
    n_detections = len(result.boxes)
    frame_detections.append(n_detections)

    if result.boxes.id is not None:
        ids = result.boxes.id.cpu().numpy().astype(int).tolist()
        all_track_ids.update(ids)

total_frames      = len(frame_detections)
frames_with_polyp = sum(1 for n in frame_detections if n > 0)
frames_empty      = total_frames - frames_with_polyp
unique_track_ids  = len(all_track_ids)
avg_per_frame     = sum(frame_detections) / total_frames

print("TRACKING RESULTS")
print("-" * 40)
print(f"Total frames processed:   {total_frames}")
print(f"Frames with detection:    {frames_with_polyp} ({frames_with_polyp/total_frames*100:.1f}%)")
print(f"Frames with no detection: {frames_empty}")
print(f"Unique track IDs assigned: {unique_track_ids}")
print(f"Avg detections per frame:  {avg_per_frame:.2f}")

In [ ]:
# Visualize Detection Rate
# Plot detections per frame to show how the model behaves
# over the temporal sequence of colonoscopy frames

fig, ax = plt.subplots(figsize=(12, 4))

ax.fill_between(range(total_frames), frame_detections,
                alpha=0.4, color="steelblue")
ax.plot(frame_detections, color="steelblue", linewidth=1)
ax.set_xlabel("Frame index")
ax.set_ylabel("Number of detections")
ax.set_title("Detections Per Frame (CVC-ClinicDB sequence)")
ax.grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(BASE_DIR, "results", "figures", "tracking_detections.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Save Tracking Stats

tracking_stats = {
    "video": {
        "input":    "cvc_input.mp4",
        "output":   "cvc_tracked.mp4",
        "fps":      VIDEO_FPS,
        "frames":   total_frames,
        "duration_sec": total_frames / VIDEO_FPS,
    },
    "detection": {
        "conf_threshold":        CONF_THRESH,
        "frames_with_detection": frames_with_polyp,
        "frames_empty":          frames_empty,
        "detection_rate":        frames_with_polyp / total_frames,
        "avg_detections_per_frame": avg_per_frame,
    },
    "tracking": {
        "unique_track_ids": unique_track_ids,
        "tracker":          "ByteTrack",
    },
}

stats_path = os.path.join(PATHS["metrics"], "tracking_stats.json")
with open(stats_path, "w") as f:
    json.dump(tracking_stats, f, indent=2)

print(f"Saved -> {stats_path}")
print()
print(json.dumps(tracking_stats, indent=2))

In [ ]:
# Summary

print("NOTEBOOK 06 COMPLETE")
print("=" * 50)
print(f"Input video:   {os.path.join(PATHS['videos'], 'cvc_input.mp4')}")
print(f"Output video:  tracked_output/ (check results/videos/)")
print()
print(f"Detection rate: {frames_with_polyp/total_frames*100:.1f}% of frames")
print(f"Unique polyp tracks: {unique_track_ids}")
print()
print("Next -> 07_streamlit_deployment")
print("  - Build the Streamlit app")
print("  - Upload/image inference with visualization")
print("  - Deploy to Streamlit Cloud")